# Content Feature Evaluation

This notebook checks what the content features are capturing and compares several text variants for recommendation quality.

## Setup

Load the same train/test split, MovieLens movie metadata, and TMDb metadata used by the recommender notebooks.

In [1]:
from pathlib import Path
import sys

current_path = Path.cwd().resolve()

for path in [current_path, *current_path.parents]:
    if (path / "src" / "data.py").exists():
        project_root = path
        break
else:
    raise FileNotFoundError("Could not find project root containing src/data.py")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd

from src.content import build_movie_content_features
from src.data import load_movies
from src.metrics import precision_at_k, recall_at_k, ndcg_at_k, rated_precision_at_k
from src.recommenders import (
    build_content_tfidf_matrix,
    recommend_content_based_movies_from_vectors,
    recommend_true_hybrid_movies_from_vectors,
)

train = pd.read_csv(project_root / "data/processed/train_ratings.csv")
test = pd.read_csv(project_root / "data/processed/test_ratings.csv")
movies = load_movies()
tmdb_metadata = pd.read_csv(project_root / "data/processed/tmdb_movie_metadata.csv")

movie_content = build_movie_content_features(
    movies=movies,
    tmdb_metadata=tmdb_metadata,
)

movie_content.head()

,movieId,title,genres,tmdb_title,overview,tagline,tmdb_genres,overview_text,overview_tagline_text,overview_genre_text,content_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,"Led by Woody, Andy's toys live happily in his ...",The adventure takes off when toys come to life!,Family|Comedy|Animation|Adventure,"Led by Woody, Andy's toys live happily in his ...","Led by Woody, Andy's toys live happily in his ...","Led by Woody, Andy's toys live happily in his ...",Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,When siblings Judy and Peter discover an encha...,It's a jungle in here.,Adventure|Fantasy|Family,When siblings Judy and Peter discover an encha...,When siblings Judy and Peter discover an encha...,When siblings Judy and Peter discover an encha...,Jumanji (1995) Adventure Children Fantasy Adve...
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,Romance|Comedy,A family wedding reignites the ancient feud be...,A family wedding reignites the ancient feud be...,A family wedding reignites the ancient feud be...,Grumpier Old Men (1995) Comedy Romance Romance...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,Comedy|Drama|Romance,"Cheated on, mistreated and stepped on, the wom...","Cheated on, mistreated and stepped on, the wom...","Cheated on, mistreated and stepped on, the wom...",Waiting to Exhale (1995) Comedy Drama Romance ...
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Just when George Banks has recovered from his ...,Just when his world is back to normal... he's ...,Comedy|Family,Just when George Banks has recovered from his ...,Just when George Banks has recovered from his ...,Just when George Banks has recovered from his ...,Father of the Bride Part II (1995) Comedy Come...


## Text Feature Variants

Compare the text fields we can use as content features. This helps separate description signal from title/genre signal.

In [2]:
text_columns = [
    "overview_text",
    "overview_tagline_text",
    "overview_genre_text",
    "content_text",
]

text_summary = []

for text_column in text_columns:
    text_lengths = movie_content[text_column].fillna("").str.split().str.len()

    text_summary.append(
        {
            "text_column": text_column,
            "avg_words": text_lengths.mean(),
            "median_words": text_lengths.median(),
            "empty_rows": (text_lengths == 0).sum(),
        }
    )

pd.DataFrame(text_summary)

,text_column,avg_words,median_words,empty_rows
0,overview_text,46.433792,42.0,122
1,overview_tagline_text,53.870355,50.0,122
2,overview_genre_text,49.820879,46.0,0
3,content_text,61.555430,58.0,0


## What Terms Does TF-IDF Capture?

For each text variant, show the highest-average TF-IDF terms across movies. This gives a quick read on whether the feature is mostly genre words, plot words, or title words.

In [3]:
def get_top_tfidf_terms(movie_content: pd.DataFrame, text_column: str, n_terms: int = 20) -> pd.DataFrame:
    vectorizer, content_vectors = build_content_tfidf_matrix(
        movie_content,
        text_column=text_column,
    )

    average_scores = content_vectors.mean(axis=0)
    average_scores = average_scores.A1

    terms = vectorizer.get_feature_names_out()

    top_indices = average_scores.argsort()[::-1][:n_terms]

    return pd.DataFrame(
        {
            "text_column": text_column,
            "term": terms[top_indices],
            "avg_tfidf": average_scores[top_indices],
        }
    )

all_top_terms = pd.concat(
    [get_top_tfidf_terms(movie_content, text_column) for text_column in text_columns],
    ignore_index=True,
)

all_top_terms

,text_column,term,avg_tfidf
0,overview_text,life,0.016893
1,overview_text,new,0.014938
2,overview_text,young,0.014722
3,overview_text,man,0.013741
4,overview_text,world,0.012566
...,...,...,...
75,content_text,new,0.014657
76,content_text,love,0.014528
77,content_text,documentary,0.014070
78,content_text,world,0.013949


## Evaluate Content Variants

Evaluate each text variant as a pure content recommender and as the content input to the balanced hybrid recommender.

In [4]:
def evaluate_recommendation_rows(recommendation_function, k: int = 10) -> dict:
    user_scores = []
    recommendation_rows = []

    positive_counts = (
        train[train["is_positive"]]
        .groupby("movieId")
        .size()
        .reset_index(name="global_positive_ratings")
    )

    for user_id in test["userId"].unique():
        recs = recommendation_function(user_id=user_id, k=k).copy()

        recommended_items = recs["movieId"].tolist()

        user_test = test[test["userId"] == user_id]
        rated_items = set(user_test["movieId"])
        positive_items = set(user_test[user_test["is_positive"]]["movieId"])

        user_scores.append(
            {
                "precision_at_10": precision_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "recall_at_10": recall_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "ndcg_at_10": ndcg_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "rated_precision_at_10": rated_precision_at_k(
                    recommended_items=recommended_items,
                    positive_items=positive_items,
                    rated_items=rated_items,
                    k=k,
                ),
            }
        )

        recs["userId"] = user_id
        recs["rank"] = range(1, len(recs) + 1)
        recommendation_rows.append(recs)

    score_results = pd.DataFrame(user_scores)
    recommendations = pd.concat(recommendation_rows, ignore_index=True)

    recommendations = recommendations.merge(
        positive_counts,
        on="movieId",
        how="left",
    )

    recommendations["global_positive_ratings"] = (
        recommendations["global_positive_ratings"].fillna(0)
    )

    return {
        "precision_at_10": score_results["precision_at_10"].mean(),
        "recall_at_10": score_results["recall_at_10"].mean(),
        "ndcg_at_10": score_results["ndcg_at_10"].mean(),
        "rated_precision_at_10": score_results["rated_precision_at_10"].mean(),
        "users_with_rated_recommendations": score_results["rated_precision_at_10"].notna().sum(),
        "unique_recommended_movies": recommendations["movieId"].nunique(),
        "catalog_coverage": recommendations["movieId"].nunique() / movies["movieId"].nunique(),
        "avg_global_positive_ratings": recommendations["global_positive_ratings"].mean(),
    }

In [5]:
content_feature_results = []

for text_column in text_columns:
    _, content_vectors = build_content_tfidf_matrix(
        movie_content,
        text_column=text_column,
    )

    pure_content_metrics = evaluate_recommendation_rows(
        lambda user_id, k: recommend_content_based_movies_from_vectors(
            train_ratings=train,
            movie_content=movie_content,
            content_vectors=content_vectors,
            user_id=user_id,
            k=k,
        )
    )

    pure_content_metrics["model"] = "pure_content"
    pure_content_metrics["text_column"] = text_column
    content_feature_results.append(pure_content_metrics)

    balanced_hybrid_metrics = evaluate_recommendation_rows(
        lambda user_id, k: recommend_true_hybrid_movies_from_vectors(
            train_ratings=train,
            movie_content=movie_content,
            content_vectors=content_vectors,
            user_id=user_id,
            k=k,
            popularity_weight=0.60,
            content_weight=0.30,
            genre_weight=0.10,
        )
    )

    balanced_hybrid_metrics["model"] = "balanced_hybrid"
    balanced_hybrid_metrics["text_column"] = text_column
    content_feature_results.append(balanced_hybrid_metrics)

content_feature_results = pd.DataFrame(content_feature_results)

content_feature_results.sort_values(
    ["model", "ndcg_at_10", "catalog_coverage"],
    ascending=[True, False, False],
)

/var/folders/7w/132kn01557v6zl_9f5zqf_t40000gn/T/ipykernel_4710/239266549.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  recommendations = pd.concat(recommendation_rows, ignore_index=True)


/var/folders/7w/132kn01557v6zl_9f5zqf_t40000gn/T/ipykernel_4710/239266549.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  recommendations = pd.concat(recommendation_rows, ignore_index=True)


/var/folders/7w/132kn01557v6zl_9f5zqf_t40000gn/T/ipykernel_4710/239266549.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  recommendations = pd.concat(recommendation_rows, ignore_index=True)


/var/folders/7w/132kn01557v6zl_9f5zqf_t40000gn/T/ipykernel_4710/239266549.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  recommendations = pd.concat(recommendation_rows, ignore_index=True)


,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,users_with_rated_recommendations,unique_recommended_movies,catalog_coverage,avg_global_positive_ratings,model,text_column
5,0.051967,0.043990,0.064451,0.771577,241,377,0.038698,124.553770,balanced_hybrid,overview_genre_text
7,0.051475,0.046471,0.063332,0.770055,241,351,0.036030,126.086230,balanced_hybrid,content_text
3,0.052459,0.042710,0.063164,0.761796,240,412,0.042291,127.336557,balanced_hybrid,overview_tagline_text
1,0.050984,0.044986,0.062826,0.752362,246,436,0.044755,125.321967,balanced_hybrid,overview_text
6,0.006066,0.007579,0.008429,0.630719,51,899,0.092281,3.902142,pure_content,content_text
0,0.006230,0.006189,0.007919,0.663462,52,1562,0.160337,5.237891,pure_content,overview_text
2,0.005738,0.005552,0.007595,0.624183,51,1483,0.152227,5.318781,pure_content,overview_tagline_text
4,0.001639,0.000394,0.001916,0.714286,14,235,0.024122,1.360791,pure_content,overview_genre_text


## Takeaways

Use this section to summarize which text source adds useful signal and whether richer content improves product tradeoffs.

In [6]:
content_feature_results.sort_values(
    ["ndcg_at_10", "catalog_coverage"],
    ascending=[False, False],
).head(10)

,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,users_with_rated_recommendations,unique_recommended_movies,catalog_coverage,avg_global_positive_ratings,model,text_column
5,0.051967,0.043990,0.064451,0.771577,241,377,0.038698,124.553770,balanced_hybrid,overview_genre_text
7,0.051475,0.046471,0.063332,0.770055,241,351,0.036030,126.086230,balanced_hybrid,content_text
3,0.052459,0.042710,0.063164,0.761796,240,412,0.042291,127.336557,balanced_hybrid,overview_tagline_text
1,0.050984,0.044986,0.062826,0.752362,246,436,0.044755,125.321967,balanced_hybrid,overview_text
6,0.006066,0.007579,0.008429,0.630719,51,899,0.092281,3.902142,pure_content,content_text
0,0.006230,0.006189,0.007919,0.663462,52,1562,0.160337,5.237891,pure_content,overview_text
2,0.005738,0.005552,0.007595,0.624183,51,1483,0.152227,5.318781,pure_content,overview_tagline_text
4,0.001639,0.000394,0.001916,0.714286,14,235,0.024122,1.360791,pure_content,overview_genre_text
